# 5-Day Educational Email Course Outline Generator

Crawls a prospect's website by gathering all relevant internal links (excluding external links, privacy and terms pages, login and signup pages, and non-content URLs), scrapes the text content from each page, and sends the synthesized content to an LLM to generate a personalized 5-day Educational Email Course outline tailored to the prospect's industry vertical, product type, and target persona. Each day of the outline uses one or two of the 10 magical ways — steps, stats, tips, lessons, mistakes, reasons, questions, personal stories, examples, or benefits — following a deliberate arc: Day 1 makes the business case using stats and reasons, Day 2 delivers a quick win using tips and steps, Day 3 goes deeper using mistakes and examples, Day 4 shifts to the human dimension using personal stories and questions, and Day 5 closes with benefits and a soft call to action. The output is a structured Markdown document with a prospect profile, five fully outlined email days each with a subject line and personalization hook, and course metadata — ready to hand to a copywriter or use directly as a brief for writing the actual email sequence.

In [ ]:
# imports

import sys
sys.path.insert(0, '..')

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from openai import OpenAI

In [ ]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

In [ ]:
# A simple web scraper to fetch text content from a site, following internal links.

EXCLUDED_PATTERNS = [
    "privacy", "terms", "cookie", "legal", "careers", 
    "login", "signup", "sign-up", "register", "logout",
    "cdn", ".pdf", ".jpg", ".png", ".zip", "mailto:"
]

def is_relevant_url(url, base_domain):
    """Filter out external, excluded, and non-content URLs."""
    parsed = urlparse(url)
    # exclude external links
    if parsed.netloc and parsed.netloc != base_domain:
        return False
    # exclude pattern matches
    if any(pattern in url.lower() for pattern in EXCLUDED_PATTERNS):
        return False
    return True

def scrape_site(start_url, max_pages=10):
    """
    Crawl a site starting from start_url.
    Returns concatenated text content from all relevant pages.
    Limits to max_pages to stay within LLM context window.
    """
    base_domain = urlparse(start_url).netloc
    visited = set()
    to_visit = [start_url]
    all_content = []

    while to_visit and len(visited) < max_pages:
        url = to_visit.pop(0)
        if url in visited:
            continue
        try:
            response = requests.get(url, timeout=10)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, "html.parser")

            # extract page text — skip nav, footer, scripts
            for tag in soup(["script", "style", "nav", 
                             "footer", "header"]):
                tag.decompose()
            page_text = soup.get_text(separator="\n", strip=True)
            all_content.append(f"## Page: {url}\n\n{page_text}\n\n")
            visited.add(url)

            # collect new links
            for link in soup.find_all("a", href=True):
                full_url = urljoin(url, link["href"])
                if (full_url not in visited 
                        and is_relevant_url(full_url, base_domain)):
                    to_visit.append(full_url)

        except Exception as e:
            print(f"Failed to fetch {url}: {e}")
            continue

    return "\n".join(all_content)
# Call the OpenAI API to generate the EEC outline
    client = OpenAI()

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": 
                user_prompt.format(website_content=website_content)}
        ]
    )

    return response.choices[0].message.content


In [ ]:
system_prompt = """You are a B2B email marketing strategist and copywriter.

Your task is to generate a personalized 5-day Educational Email Course (EEC) outline
for a SaaS prospect, based on scraped content from their website.

The outline must follow this deliberate arc:
- Day 1: Make the business case — use STATS and REASONS
- Day 2: Deliver a quick win — use TIPS and STEPS
- Day 3: Go deeper — use MISTAKES and EXAMPLES
- Day 4: The human dimension — use PERSONAL STORIES and QUESTIONS
- Day 5: Close strong — use BENEFITS and a soft CALL TO ACTION

Output a structured Markdown document with:
1. **Prospect Profile** — industry vertical, product type, target persona (inferred from the website)
2. **Five Email Days** — each with:
   - Day number and theme
   - Subject line (compelling, specific to the prospect)
   - Personalization hook (one sentence tying the email to the prospect's world)
   - Content outline (3-5 bullet points using the day's magical ways)
3. **Course Metadata** — recommended send cadence, tone, and key differentiator to weave throughout

Be specific. Reference details from the website content. Avoid generic advice."""

user_prompt = """Here is the scraped content from the prospect's website:

{website_content}

Based on this content, generate a personalized 5-day Educational Email Course outline
following the structure in your instructions."""

In [ ]:
# Generate a personalized 5-day EEC outline for a website
def generate_eec_outline(website_url):
    """
    Scrape a website and generate a personalized 
    5-day EEC outline using an LLM.
    """
    print(f"Scraping {website_url}...")
    website_content = scrape_site(website_url)

    # truncate to stay within context window
    # roughly 12,000 words — adjust based on model
    max_chars = 48000
    if len(website_content) > max_chars:
        website_content = website_content[:max_chars] + \
            "\n\n[Content truncated to fit context window]"
            

    print(f"Scraped {len(website_content)} characters. Generating outline...")
    # Call the OpenAI API to generate the EEC outline
    client = OpenAI()

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": 
                user_prompt.format(website_content=website_content)}
        ]
    )

    return response.choices[0].message.content


In [ ]:
# A function to display this nicely in the output, using markdown

def display_outline(url):
    generated_outline = generate_eec_outline(url)

    display(Markdown(generated_outline))

In [ ]:
# Display the outline for the w3C's inaccessibility demo site
displayed_outline = display_outline("https://www.w3.org/WAI/demos/bad/before/home.html")
generated_outline

In [ ]:
# optionally save to file
filename = urlparse(url).netloc.replace(".", "_") + "_eec_outline.md"
with open(filename, "w") as f:
    f.write(outline)
print(f"\nSaved to {filename}")